# 🧑‍🏫 Regression Microprojects — Instructor Notebook
*Elements of Data Science* <br>
After Lab 8, each student applies their **regression toolbox** (`standard_units`, `correlation`, `slope`, `intercept`) to a new, real data set with much less scaffolding. This notebook:

1. Reads the **class list** and **randomly assigns** each student to one of five microprojects
2. Saves the assignments to `shared-public` so each student's notebook can confirm their project with `my_project(name)`
3. Shows a live **gallery** of the figures students submit with `submit_plot(...)`, plus a table of their numbers for class discussion

|Project|Data|Twist|Typical *r*|
|---|---|---|---|
|MP1 🌍 CO₂ at Mauna Loa|NOAA annual means, 1959–2025|Errors curve: the rise is accelerating; extrapolation|0.99|
|MP2 📏 Galton's Heights|934 children, 1886|Regression to the mean; gender as a hidden variable|0.32|
|MP3 👶 Birth Weight|1,174 births (CHDS)|Smokers vs non-smokers at the same gestation; outliers|0.41|
|MP4 🐧 Palmer Penguins|342 penguins, 3 species|Overall *r* > every within-species *r*|0.87|
|MP5 ⚛️ Periodic Trends|83 elements|Outlier elements; tighter trend within a period; common cause|0.90|

**Files:** copy `microproject_tools.py` into `/home/jovyan/shared-public/`. Student notebooks and their CSVs go together in one folder (distributed the same way as the labs). Answer keys are in `keys/` — keep them out of the student distribution.

## ⚙️ Setup

In [ ]:
import sys, os, glob, json
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "shared-public")))
sys.path.insert(0, "/home/jovyan/shared-public")
from init import *
import microproject_tools as mpt

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
# Where everything lives (set in microproject_tools.py)
os.makedirs(mpt.SUBMIT_DIR, exist_ok=True)
try:
    os.chmod(mpt.MICRO_ROOT, 0o777); os.chmod(mpt.SUBMIT_DIR, 0o777)   # let every student write submissions
except PermissionError:
    pass
print('Assignments file :', mpt.ASSIGN_FILE)
print('Submissions folder:', mpt.SUBMIT_DIR)

projects = Table().with_columns(
    'project',  make_array(*mpt.PROJECTS.keys()),
    'title',    make_array(*[v[0] for v in mpt.PROJECTS.values()]),
    'notebook', make_array(*[v[1] for v in mpt.PROJECTS.values()]))
projects

## 1. Class List
`class_list.csv` needs a **`first_name`** column; a `last_name` column is optional but lets the code tell apart students who share a first name. (From a Canvas gradebook export, split the *Student* column, which is formatted `Last, First`.)

In [ ]:
roster = Table.read_table('class_list.csv')
first_names = roster.column('first_name')
if 'last_name' in roster.labels:
    last_initials = make_array(*[str(s)[:1].upper() for s in roster.column('last_name')])
else:
    last_initials = make_array(*([''] * roster.num_rows))

# Students who share a first name are shown as 'Alex K.' and 'Alex M.'
display_names = make_array()
for i in np.arange(roster.num_rows):
    shared = np.count_nonzero(first_names == first_names.item(i)) > 1
    label = f'{first_names.item(i)} {last_initials.item(i)}.' if shared and last_initials.item(i) else first_names.item(i)
    display_names = np.append(display_names, label)

print(roster.num_rows, 'students')
duplicates = np.unique(display_names)[np.unique(display_names, return_counts=True)[1] > 1]
if len(duplicates):
    print('⚠️ Still ambiguous — add last names for:', duplicates)

## 2. Random Assignment
We shuffle the class with `np.random.choice(..., replace=False)` — a random **permutation** — and then deal students out to the projects like cards, so the groups differ in size by at most one. The project order is shuffled too, so no project always gets the extra student.

Setting a **seed** makes the assignment reproducible: rerun with the same seed and you get the same groups. Change the seed to re-deal.

In [ ]:
seed = 1013
np.random.seed(seed)

n = len(display_names)
shuffled = np.random.choice(np.arange(n), n, replace=False)              # a random order of the students
project_order = np.random.choice(projects.column('project'), projects.num_rows, replace=False)
dealt = project_order.take(np.arange(n) % projects.num_rows)            # deal like cards: MP?, MP?, ... repeat

assignments = Table().with_columns(
    'display_name', display_names.take(shuffled),
    'first_name',   first_names.take(shuffled),
    'last_initial', last_initials.take(shuffled),
    'project',      dealt).sort('project')
assignments.group('project').join('project', projects.select('project', 'title'))

Each group, ready to show on screen:

In [ ]:
for pid in projects.column('project'):
    names = assignments.where('project', pid).column('display_name')
    print(f"{pid} — {mpt.PROJECTS.get(pid, ('Instructor demo',))[0]:28s}  {', '.join(sorted(names))}")

### Save the assignments
Students' `my_project(name)` reads this file. To protect a deal that students have already seen, the cell refuses to overwrite an existing file unless you set `overwrite = True`.

In [ ]:
overwrite = False

if os.path.exists(mpt.ASSIGN_FILE) and not overwrite:
    print('Assignments already saved — set overwrite = True to replace them.')
    print(Table.read_table(mpt.ASSIGN_FILE).group('project'))
else:
    assignments.to_csv(mpt.ASSIGN_FILE)
    try:
        os.chmod(mpt.ASSIGN_FILE, 0o644)
    except PermissionError:
        pass
    print('Saved', assignments.num_rows, 'assignments to', mpt.ASSIGN_FILE)

## 3. Submission Gallery 🖼️
Rerun the cells below any time during class. Each submission is a PNG plus a small JSON file with the student's *r*, slope, intercept, and caption. Resubmissions replace older versions.

In [ ]:
def load_submissions():
    "Read every submission record into a Table."
    records = [json.load(open(p)) for p in sorted(glob.glob(os.path.join(mpt.SUBMIT_DIR, '*.json')))]
    stems = [os.path.basename(p)[:-5] for p in sorted(glob.glob(os.path.join(mpt.SUBMIT_DIR, '*.json')))]
    if not records:
        return Table().with_columns('project', make_array(), 'name', make_array())
    return Table().with_columns(
        'project',   make_array(*[r['project'] for r in records]),
        'name',      make_array(*[r['name'] for r in records]),
        'first',     make_array(*[r['first_name'].lower() for r in records]),
        'r',         make_array(*[r['r'] for r in records]),
        'slope',     make_array(*[r['slope'] for r in records]),
        'intercept', make_array(*[r['intercept'] for r in records]),
        'caption',   make_array(*[r['caption'] for r in records]),
        'submitted', make_array(*[r['submitted'] for r in records]),
        'warnings',  make_array(*[len(r.get('figure_warnings', [])) for r in records]),
        'png',       make_array(*[os.path.join(mpt.SUBMIT_DIR, s + '.png') for s in stems]))

submissions = load_submissions()
print(submissions.num_rows, 'submissions')
submissions.drop('png', 'first').sort('project')

### Who has submitted?

In [ ]:
roster_status = assignments.with_columns(
    'submitted', make_array(*['✅' if str(f).lower() in submissions.column('first') else '—'
                              for f in assignments.column('first_name')]) if submissions.num_rows
                 else make_array(*(['—'] * assignments.num_rows)))
print(np.count_nonzero(roster_status.column('submitted') == '✅'), 'of', roster_status.num_rows, 'students have submitted')
roster_status.select('project', 'display_name', 'submitted').show()

### Show one project's figures

In [ ]:
def show_gallery(pid, cols=2):
    "Display every submitted figure for one project, with the student's caption."
    sub = submissions.where('project', pid) if submissions.num_rows else submissions
    print(f'{pid}: {mpt.PROJECTS.get(pid, ('Instructor demo',))[0]} — {sub.num_rows} figures')
    if sub.num_rows == 0:
        return
    rows = int(np.ceil(sub.num_rows / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(8 * cols, 5.2 * rows), squeeze=False)
    for ax in axes.flatten():
        ax.axis('off')
    for i in np.arange(sub.num_rows):
        ax = axes.flatten()[i]
        ax.imshow(plt.imread(sub.column('png').item(i)))
        ax.set_title(sub.column('name').item(i), fontsize=14, weight='bold')
        caption = sub.column('caption').item(i)
        ax.text(0.5, -0.04, caption, transform=ax.transAxes, ha='center', va='top', fontsize=10, wrap=True)
    plt.tight_layout()
    plt.show()

show_gallery('MP1')

In [ ]:
for pid in projects.column('project'):
    show_gallery(pid)

## 4. Discussion: Did Everyone Get the Same Line?
Students on the same project used the same data, so their full-data *r* and slope **should agree**. Disagreement is a great discussion starter: a bug in someone's toolbox, a subset of the data (the twist line instead of the full-data line), or swapped $x$ and $y$.

Across projects, compare *r* with how good the fit *looked*: MP1 has the largest *r* but a clearly curved error plot; MP2 has the smallest *r* yet tells the oldest story in regression ([Ch. 15.2](https://inferentialthinking.com/chapters/15/2/Regression_Line.html)).

In [ ]:
if submissions.num_rows:
    summary = submissions.select('project', 'r', 'slope').group('project', np.mean).join(
        'project', submissions.select('project', 'r').group('project', np.std).relabel('r std', 'spread of r'))
    summary.show()

**Prompts for the gallery walk**
- Which figure would you put in a newspaper? Why?
- Which project shows that a large *r* does not guarantee a straight line is the right model?
- In which projects did splitting the data into groups change the story?
- Which findings could you state as **causes**, and which only as **associations**?